# 3 · Supervisor (Multi-Agent Coordination)

Wrap each specialist as a **tool** and give them to a **supervisor** agent that decides who handles
what — the agents-as-tools pattern from wk4.1.

```mermaid
graph TD
    S[Supervisor] -->|policy| K[Knowledge agent]
    S -->|orders| O[Order agent]
    S -->|shipping| W[Web agent]
```

In [1]:
%run aurora_common.py

C:\Users\akhawaja\git\cs4603\wk4_capstone\aurora_common.py:42: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Build the specialists (compact versions of notebook 2)

In [2]:
retriever = get_knowledge_retriever(k=3)
orders_db = get_orders_sqldatabase()


@tool
def search_help_center(query: str) -> str:
    """Search store help-center policies."""
    return "\n\n".join(d.page_content for d in retriever.invoke(query))


@tool
def run_sql(query: str) -> str:
    """Run a read-only SQL query against the orders database."""
    try:
        return orders_db.run(query)
    except Exception as e:
        return f"Error: {e}"


@tool
def track_shipment(order_id: str) -> str:
    """Look up carrier tracking status for an order (simulated)."""
    return f"Order {order_id}: in transit, expected in 2 days (carrier: ACME Express)."


knowledge_agent = create_agent(model=llm_noreason, tools=[search_help_center],
                               system_prompt="Answer store-policy questions using the help-center tool.")
order_agent = create_agent(model=llm_noreason, tools=[run_sql],
                           system_prompt="Answer order questions with SELECT queries. Tables: customers, products, orders, order_items.")
web_agent = create_agent(model=llm_noreason, tools=[track_shipment],
                         system_prompt="Look up shipment/carrier info.")

## Wrap each specialist as a tool, then build the supervisor

In [3]:
@tool
def ask_knowledge_agent(question: str) -> str:
    """Delegate a store-policy question to the knowledge (RAG) agent."""
    r = knowledge_agent.invoke({"messages": [HumanMessage(content=question)]})
    return r["messages"][-1].content


@tool
def ask_order_agent(question: str) -> str:
    """Delegate an order/customer data question to the SQL order agent."""
    r = order_agent.invoke({"messages": [HumanMessage(content=question)]})
    return r["messages"][-1].content


@tool
def ask_web_agent(question: str) -> str:
    """Delegate a shipment/tracking question to the web agent."""
    r = web_agent.invoke({"messages": [HumanMessage(content=question)]})
    return r["messages"][-1].content


supervisor = create_agent(
    model=llm,
    tools=[ask_knowledge_agent, ask_order_agent, ask_web_agent],
    system_prompt=(
        "You are a support supervisor. Route each request to the right specialist tool:\n"
        "- policy/returns/refund rules -> ask_knowledge_agent\n"
        "- order/customer data -> ask_order_agent\n"
        "- shipment/tracking -> ask_web_agent\n"
        "Combine their answers into one helpful reply."
    ),
)

## Test routing
> **TODO (student):** feed the `route_to` value from notebook 1's routing chain to pre-select the
> specialist instead of relying only on the supervisor's judgement.

In [4]:
for q in [
    "What is the status of order 1001?",
    "How many days do I have to return opened electronics?",
    "Where is my order 1001 right now?",
]:
    print("Q:", q)
    r = supervisor.invoke({"messages": [HumanMessage(content=q)]})
    print("A:", r["messages"][-1].content)
    print()

Q: What is the status of order 1001?


A: The status of order 1001 is **shipped**.

Q: How many days do I have to return opened electronics?


A: You have **14 days** from the date of delivery to return opened electronics.

Please keep the following in mind:
*   **Premium Members:** If you have a Premium membership, your return window is extended to **60 days**.
*   **Condition:** While returns are accepted for opened items, they generally need to be in their original packaging and in good condition.

If you need help initiating a return or have questions about your specific order, feel free to ask!

Q: Where is my order 1001 right now?


A: [{'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': "The user is asking about the status of their order 1001. This is a shipment/tracking question, so I should use the ask_web_agent tool. I've already called it and received the response. Now I need to provide a helpful reply to the user with this information."}]}, {'type': 'text', 'text': 'Your order 1001 is currently in transit with ACME Express and is expected to arrive in 2 days.'}]



### Definition of done
- Supervisor routes policy / order / shipping questions to the correct specialist.
- Answers are coherent and combine specialist output.